In [ ]:
# DS_GEN — CELL 1: Select an industry and use case, then run Cell 2.
# The choices and targets are shown below. No typing/case-name guessing needed.
from pathlib import Path

ROWS = 1000
SEED = 42
NOISE = 0.25
MISSING_RATE = 0.02
RUN_MODEL = True
OUTPUT_DIR = '/kaggle/working' if Path('/kaggle/working').is_dir() else './kaggle_output'

# Single source of truth for every available industry and use case.
# To expand later, add choices here and implement their generation rules in Cell 2.
CATALOG = {'airlines': ('flight_delays', 'maintenance', 'flight_cancellations'),
 'banking': ('transaction_fraud', 'loan_default', 'credit_risk'),
 'healthcare': ('appointment_no_show', 'readmission_risk'),
 'retail': ('customer_churn', 'demand_forecast', 'return_risk'),
 'manufacturing': ('equipment_failure', 'quality_defect'),
 'logistics': ('late_delivery', 'shipment_damage'),
 'cybersecurity': ('suspicious_login', 'alert_severity', 'account_takeover'),
 'cloud_ai': ('high_latency', 'capacity_forecast', 'gpu_cost_forecast'),
 'energy': ('peak_load', 'equipment_outage'),
 'real_estate': ('property_price', 'long_listing'),
 'education': ('student_dropout', 'exam_score'),
 'agriculture': ('crop_yield', 'irrigation_failure'),
 'hospitality': ('booking_cancellation', 'nightly_rate'),
 'insurance': ('claim_fraud', 'claim_cost')}
TARGETS = {'flight_delays': 'delay_minutes',
 'maintenance': 'needs_maintenance',
 'flight_cancellations': 'cancelled',
 'transaction_fraud': 'is_fraud',
 'loan_default': 'loan_default',
 'credit_risk': 'high_credit_risk',
 'appointment_no_show': 'no_show',
 'readmission_risk': 'readmitted_30d',
 'customer_churn': 'churned',
 'demand_forecast': 'units_demanded',
 'return_risk': 'returned',
 'equipment_failure': 'failure_next_7d',
 'quality_defect': 'defect_found',
 'late_delivery': 'late_delivery',
 'shipment_damage': 'damaged',
 'suspicious_login': 'suspicious_login',
 'alert_severity': 'severity_level',
 'account_takeover': 'account_takeover',
 'high_latency': 'high_latency',
 'capacity_forecast': 'requests_next_hour',
 'gpu_cost_forecast': 'gpu_cost_usd',
 'peak_load': 'demand_kw',
 'equipment_outage': 'outage_next_30d',
 'property_price': 'price_usd',
 'long_listing': 'listing_over_90d',
 'student_dropout': 'dropped_out',
 'exam_score': 'exam_score',
 'crop_yield': 'yield_tons_hectare',
 'irrigation_failure': 'irrigation_failure',
 'booking_cancellation': 'cancelled',
 'nightly_rate': 'nightly_rate_usd',
 'claim_fraud': 'is_fraudulent_claim',
 'claim_cost': 'claim_cost_usd'}

INDUSTRY = 'airlines'   # Fallback if widgets are unavailable
USE_CASE = 'flight_delays'

print(f"Available: {len(CATALOG)} industries / {sum(map(len, CATALOG.values()))} use cases")
for industry, cases in CATALOG.items():
    print(f"  {industry}: " + ', '.join(f"{name} [{TARGETS[name]}]" for name in cases))

try:
    import ipywidgets as widgets
    from IPython.display import display
    industry_picker = widgets.Dropdown(options=list(CATALOG), value=INDUSTRY, description='Industry:', style={'description_width':'initial'}, layout=widgets.Layout(width='360px'))
    case_picker = widgets.Dropdown(options=[(f"{c} → {TARGETS[c]}",c) for c in CATALOG[INDUSTRY]], value=USE_CASE, description='Use case:', style={'description_width':'initial'}, layout=widgets.Layout(width='460px'))
    selection_info = widgets.HTML()
    def update_selection(change=None):
        cases = CATALOG[industry_picker.value]
        current = case_picker.value
        case_picker.options = [(f"{c} → {TARGETS[c]}",c) for c in cases]
        if current not in cases:
            case_picker.value = cases[0]
        selection_info.value = f"<b>Selected:</b> {industry_picker.value} / {case_picker.value} &nbsp; <b>Target:</b> {TARGETS[case_picker.value]}"
    industry_picker.observe(update_selection, names='value')
    case_picker.observe(update_selection, names='value')
    update_selection()
    display(widgets.VBox([widgets.HTML('<b>Choose your dataset, then run Cell 2 to generate it.</b>'), industry_picker, case_picker, selection_info]))
except ImportError:
    print('ipywidgets unavailable; edit INDUSTRY and USE_CASE above, then run Cell 2.')


In [ ]:
import json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error



# CATALOG and TARGETS are defined in Cell 1 (also powers dropdowns).


# Five extensions reuse existing industry features; no duplicated generators.
# (base case, target column, task)
EXTRA_CASES = {
    'flight_cancellations': ('flight_delays', 'cancelled', 'classification'),
    'credit_risk': ('loan_default', 'high_credit_risk', 'classification'),
    'return_risk': ('customer_churn', 'returned', 'classification'),
    'account_takeover': ('suspicious_login', 'account_takeover', 'classification'),
    'gpu_cost_forecast': ('capacity_forecast', 'gpu_cost_usd', 'regression'),
}


# Compact schemas for additional industries: numeric fields (low, high), categories,
# target formula weights; no separate generator function per use case.
NEW_SCHEMAS = {
 'education': {
   'numeric': {'attendance_pct': (45, 100), 'study_hours_week': (1, 35), 'prior_gpa': (1.5, 4), 'missed_assignments': (0, 15)},
   'categorical': {'program': ['science','arts','business']},
   'cases': {'student_dropout': ('dropped_out', 'classification', [-.055,-.08,-1.3,.18], 7.5),
             'exam_score': ('exam_score', 'regression', [.55,.65,9,-1.1], -20)}},
 'agriculture': {
   'numeric': {'rainfall_mm': (80, 800), 'soil_quality': (0, 1), 'fertilizer_kg_ha': (10, 220), 'temp_c': (12, 41)},
   'categorical': {'crop_type': ['corn','wheat','soy','rice']},
   'cases': {'crop_yield': ('yield_tons_hectare', 'regression', [.003,3,.007,-.07], 1.5),
             'irrigation_failure': ('irrigation_failure', 'classification', [-.003,-2,-.003,.09], -1)}},
 'hospitality': {
   'numeric': {'lead_days': (0, 220), 'night_stay': (1, 14), 'prior_cancellations': (0, 6), 'season_demand': (0, 1)},
   'categorical': {'room_type': ['standard','deluxe','suite']},
   'cases': {'booking_cancellation': ('cancelled', 'classification', [.014,-.09,.6,.3], -2.5),
             'nightly_rate': ('nightly_rate_usd', 'regression', [.10,1.5,-1,145], 75)}},
 'insurance': {
   'numeric': {'claim_amount_usd': (300, 25000), 'policy_age_months': (1, 180), 'prior_claims': (0, 8), 'risk_score': (0, 1)},
   'categorical': {'claim_type': ['auto','property','liability']},
   'cases': {'claim_fraud': ('is_fraudulent_claim', 'classification', [.00006,-.008,.25,2.1], -3.3),
             'claim_cost': ('claim_cost_usd', 'regression', [.72,-1.2,130,1500], 300)}}
}

def generate_from_schema(industry, use_case, n, seed, noise):
    spec = NEW_SCHEMAS[industry]
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({'record_id': [f'{industry[:3].upper()}-{i:06d}' for i in range(n)]})
    for name, (low, high) in spec['numeric'].items():
        df[name] = rng.uniform(low, high, n).round(3)
    for name, options in spec['categorical'].items():
        df[name] = rng.choice(options, n)
    target, task, weights, intercept = spec['cases'][use_case]
    values = df[list(spec['numeric'])].to_numpy(dtype=float)
    signal = intercept + values @ np.asarray(weights)
    if task == 'classification':
        df[target] = (rng.random(n) < sigmoid(signal + rng.normal(0, noise, n))).astype(int)
    else:
        df[target] = np.maximum(0, signal + rng.normal(0, max(1, np.std(signal)*noise), n)).round(2)
    return df

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -40, 40)))

def generate(industry, use_case, n, seed, noise):
    if industry in NEW_SCHEMAS:
        return generate_from_schema(industry, use_case, n, seed, noise)
    if use_case in EXTRA_CASES:
        base, target, task = EXTRA_CASES[use_case]
        df = generate(industry, base, n, seed, noise)
        rng = np.random.default_rng(seed + 1009)
        jitter = rng.normal(0, noise, n)
        if use_case == 'flight_cancellations':
            p = sigmoid(-5 + .065*df['delay_minutes'] + 2*df['weather_severity'] + jitter)
        elif use_case == 'credit_risk':
            p = sigmoid(-4 + 4*df['debt_to_income'] + 3*df['credit_utilization'] + jitter)
        elif use_case == 'return_risk':
            p = sigmoid(-3 + .18*df['support_tickets'] + .004*df['monthly_spend_usd'] + jitter)
        elif use_case == 'account_takeover':
            p = sigmoid(-5 + .18*df['failed_attempts'] + 1.5*df['new_device'] + .0002*df['geo_distance_km'] + jitter)
        else:
            df[target] = np.maximum(0, (.045*df['gpu_util_pct'] + .002*df['input_tokens']
                                + .025*df['concurrency'] + .1*df['queue_depth'])
                                * (1 + jitter*.1)).round(2)
        if task == 'classification':
            df[target] = (rng.random(n) < p).astype(int)
        # Base label is removed to avoid target leakage.
        base_target = {
            'flight_delays':'delay_minutes', 'loan_default':'loan_default',
            'customer_churn':'churned', 'suspicious_login':'suspicious_login',
            'capacity_forecast':'requests_next_hour'
        }[base]
        return df.drop(columns=[base_target])
    rng = np.random.default_rng(seed)
    r = rng
    integer = lambda a,b: r.integers(a,b+1,size=n)
    choose = lambda x: r.choice(x,size=n)
    normal = lambda mu,sd: r.normal(mu,sd,size=n)
    indicator = lambda p: (r.random(n)<np.clip(p,0,1)).astype(int)
    df = pd.DataFrame({'record_id':[f'{industry[:3].upper()}-{i:06d}' for i in range(n)]})
    if industry=='airlines':
        df['airline']=choose(['AA','DL','UA','WN']); df['route_type']=choose(['domestic','international'])
        df['distance_miles']=integer(150,3200); df['weather_severity']=r.beta(2,5,n).round(3)
        df['airport_congestion']=r.uniform(0,1,n).round(3); df['departure_hour']=integer(0,23)
        df['inbound_delay_min']=r.exponential(12,n).round(1)
        df['flight_note']=np.where(df.weather_severity>.65,'severe weather advisory',np.where(df.airport_congestion>.72,'ground operations congested','scheduled operations normal'))
        if use_case=='flight_delays':
            y=np.maximum(0,5+65*df.weather_severity+40*df.airport_congestion+0.55*df.inbound_delay_min+normal(0,12+40*noise))
            df['delay_minutes']=y.round(1)
        else:
            df['aircraft_age_years']=integer(1,29); df['cycles_since_service']=integer(10,1300)
            df['maintenance_note']=np.where(df.cycles_since_service>950,'inspection recommended','routine service history')
            p=sigmoid(-5+0.11*df.aircraft_age_years+0.003*df.cycles_since_service+normal(0,noise))
            df['needs_maintenance']=indicator(p)
    elif industry=='banking':
        df['channel']=choose(['mobile','web','branch','atm']); df['account_age_months']=integer(1,240)
        df['amount_usd']=r.lognormal(4.3,1.1,n).round(2); df['transactions_24h']=integer(1,30)
        df['international']=indicator(.12); df['customer_note']=np.where(df.international==1,'cross-border activity','domestic account activity')
        if use_case=='transaction_fraud':
            df['merchant_category']=choose(['retail','fuel','travel','online','groceries'])
            p=sigmoid(-5+0.75*np.log1p(df.amount_usd)+.07*df.transactions_24h+.9*df.international+normal(0,noise))
            df['is_fraud']=indicator(p)
            df['transaction_description']=np.where(df.international==1,'international payment request','domestic merchant purchase')
        else:
            df['annual_income_usd']=integer(28000,220000); df['debt_to_income']=r.uniform(.05,.85,n).round(3)
            df['credit_utilization']=r.uniform(.01,.99,n).round(3)
            p=sigmoid(-4+.025*(df.debt_to_income*100)+.018*(df.credit_utilization*100)-.000012*df.annual_income_usd+normal(0,noise))
            df['loan_default']=indicator(p)
    elif industry=='healthcare':
        df['age']=integer(18,90); df['prior_visits']=integer(0,12)
        df['travel_minutes']=integer(5,100); df['appointment_type']=choose(['routine','follow_up','specialist'])
        df['visit_note']=np.where(df.prior_visits>6,'multiple prior encounters','standard appointment history')
        if use_case=='appointment_no_show':
            df['days_until_appointment']=integer(1,90)
            p=sigmoid(-2+.018*df.travel_minutes+.016*df.days_until_appointment-.09*df.prior_visits+normal(0,noise))
            df['no_show']=indicator(p)
        else:
            df['length_of_stay_days']=integer(1,15); df['comorbidity_count']=integer(0,7)
            df['discharge_note']=np.where(df.comorbidity_count>3,'follow-up recommended','routine discharge plan')
            p=sigmoid(-4+.026*df.age+.3*df.comorbidity_count+.065*df.length_of_stay_days+normal(0,noise))
            df['readmitted_30d']=indicator(p)
    elif industry=='retail':
        df['region']=choose(['north','south','east','west']); df['customer_tenure_months']=integer(1,120)
        df['monthly_spend_usd']=r.lognormal(4.4,.7,n).round(2); df['support_tickets']=integer(0,12)
        df['customer_feedback']=np.where(df.support_tickets>7,'frequent support complaints','generally satisfactory service')
        if use_case=='customer_churn':
            p=sigmoid(-1.5+.3*df.support_tickets-.018*df.customer_tenure_months-.002*df.monthly_spend_usd+normal(0,noise))
            df['churned']=indicator(p)
        else:
            df['promotion']=indicator(.25); df['season_index']=r.uniform(.6,1.5,n).round(3)
            demand=np.maximum(0,12+.11*df.monthly_spend_usd+22*df.promotion+38*df.season_index+normal(0,10+25*noise))
            df['units_demanded']=np.round(demand).astype(int)
    elif industry=='manufacturing':
        df['machine_type']=choose(['press','cnc','packaging','robot']); df['machine_age_years']=integer(0,25)
        df['temperature_c']=normal(72,15).round(1); df['vibration_mm_s']=r.gamma(2.3,1.5,n).round(2)
        df['operator_note']=np.where(df.vibration_mm_s>6,'elevated vibration detected','routine operating range')
        if use_case=='equipment_failure':
            df['hours_since_service']=integer(10,3000)
            p=sigmoid(-6+.07*df.machine_age_years+.35*df.vibration_mm_s+.0008*df.hours_since_service+.025*(df.temperature_c-72)+normal(0,noise))
            df['failure_next_7d']=indicator(p)
        else:
            df['production_speed']=integer(20,250)
            p=sigmoid(-5+.03*(df.temperature_c-72)+.32*df.vibration_mm_s+.007*df.production_speed+normal(0,noise))
            df['defect_found']=indicator(p)
    elif industry=='logistics':
        df['shipping_mode']=choose(['road','air','rail','sea']); df['distance_km']=integer(20,4500)
        df['weather_severity']=r.uniform(0,1,n).round(3); df['warehouse_backlog']=integer(0,100)
        df['shipment_note']=np.where(df.weather_severity>.7,'weather disruption reported','transit progressing normally')
        if use_case=='late_delivery':
            p=sigmoid(-4+.9*df.weather_severity+.026*df.warehouse_backlog+.0004*df.distance_km+normal(0,noise))
            df['late_delivery']=indicator(p)
        else:
            df['fragile']=indicator(.24)
            p=sigmoid(-4+1.1*df.fragile+.7*df.weather_severity+.007*df.distance_km+normal(0,noise))
            df['damaged']=indicator(p)
    elif industry=='cybersecurity':
        df['login_hour']=integer(0,23); df['failed_attempts']=integer(0,20)
        df['new_device']=indicator(.25); df['geo_distance_km']=integer(0,9000)
        df['security_event']=np.where(df.failed_attempts>10,'repeated authentication failures','standard authentication activity')
        if use_case=='suspicious_login':
            p=sigmoid(-4+.19*df.failed_attempts+1.15*df.new_device+.00017*df.geo_distance_km+normal(0,noise))
            df['suspicious_login']=indicator(p)
        else:
            risk=.2*df.failed_attempts+2*df.new_device+df.geo_distance_km/2200+normal(0,1+noise)
            df['severity_level']=pd.cut(risk,[-np.inf,2,5,np.inf],labels=['low','medium','high']).astype(str)
    elif industry=='cloud_ai':
        df['model_family']=choose(['transformer','cnn','rnn','xgboost']); df['concurrency']=integer(1,300)
        df['queue_depth']=r.poisson(10,n); df['gpu_util_pct']=r.uniform(10,99,n).round(1)
        df['input_tokens']=integer(50,6000)
        df['service_log']=np.where(df.queue_depth>15,'queue pressure observed','normal serving operation')
        if use_case=='high_latency':
            lat=15+.4*df.concurrency+3.8*df.queue_depth+.012*df.input_tokens+normal(0,9+25*noise)
            df['latency_ms']=np.maximum(0,lat).round(1)
            df['high_latency']= (df.latency_ms>190).astype(int)
        else:
            df['requests_next_hour']=np.maximum(0,(25+2*df.concurrency+.8*df.queue_depth+normal(0,20+40*noise))).round().astype(int)
    elif industry=='energy':
        df['region']=choose(['metro','suburban','rural']); df['temperature_c']=normal(24,12).round(1)
        df['hour']=integer(0,23); df['households']=integer(50,5000)
        df['grid_note']=np.where(df.temperature_c>33,'high cooling demand expected','typical grid conditions')
        if use_case=='peak_load':
            demand=np.maximum(0,.8*df.households+15*np.abs(df.temperature_c-20)+80*(df.hour.between(16,20))+normal(0,40+60*noise))
            df['demand_kw']=demand.round(1)
        else:
            df['transformer_age_years']=integer(1,40)
            p=sigmoid(-6+.08*df.transformer_age_years+.04*np.abs(df.temperature_c-20)+.0002*df.households+normal(0,noise))
            df['outage_next_30d']=indicator(p)
    else: # real_estate
        df['property_type']=choose(['house','condo','townhome']); df['sqft']=integer(600,4800)
        df['bedrooms']=integer(1,6); df['property_age_years']=integer(0,90)
        df['distance_to_center_km']=r.uniform(1,65,n).round(1)
        df['listing_description']=np.where(df.sqft>2500,'spacious property with flexible layout','comfortable residence with practical floorplan')
        if use_case=='property_price':
            price=np.maximum(40000,45000+185*df.sqft+16000*df.bedrooms-1300*df.property_age_years-850*df.distance_to_center_km+normal(0,14000+35000*noise))
            df['price_usd']=price.round(0).astype(int)
        else:
            df['asking_price_usd']=integer(130000,950000)
            p=sigmoid(-3+.000003*df.asking_price_usd+.025*df.distance_to_center_km-.0003*df.sqft+normal(0,noise))
            df['listing_over_90d']=indicator(p)
    return df


def add_missing(df, rate, seed):
    if not 0<=rate<1: raise ValueError('MISSING_RATE must be in [0,1)')
    out=df.copy(); rng=np.random.default_rng(seed+999)
    protected={'record_id', 'target'}
    outcome = {
      'delay_minutes','needs_maintenance','is_fraud','loan_default','no_show','readmitted_30d','churned',
      'units_demanded','failure_next_7d','defect_found','late_delivery','damaged','suspicious_login',
      'severity_level','latency_ms','high_latency','requests_next_hour','demand_kw','outage_next_30d',
      'price_usd','listing_over_90d'
    }
    protected |= outcome | set(TARGETS.values())
    for col in out.columns:
        if col not in protected:
            out.loc[rng.random(len(out))<rate,col]=np.nan
    return out


def run_baseline(df, target, seed):
    X=df.drop(columns=[target,'record_id']+(['latency_ms'] if target=='high_latency' else []),errors='ignore')
    y=df[target]
    numeric=X.select_dtypes(include='number').columns.tolist()
    categorical=X.columns.difference(numeric).tolist()
    prep=ColumnTransformer([
       ('num',SimpleImputer(strategy='median',add_indicator=True),numeric),
       ('cat',Pipeline([('fill',SimpleImputer(strategy='most_frequent')),
                        ('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical)
    ])
    regression_targets={'delay_minutes','units_demanded','latency_ms','requests_next_hour','demand_kw','price_usd','exam_score','yield_tons_hectare','nightly_rate_usd','claim_cost_usd','gpu_cost_usd'}
    is_class=target not in regression_targets
    estimator=RandomForestClassifier(n_estimators=60,random_state=seed,n_jobs=-1) if is_class else RandomForestRegressor(n_estimators=60,random_state=seed,n_jobs=-1)
    pipeline=Pipeline([('preprocess',prep),('model',estimator)])
    stratify=y if is_class and y.value_counts().min()>=2 else None
    try: xtr,xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=seed,stratify=stratify)
    except ValueError: xtr,xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=seed)
    pipeline.fit(xtr,ytr)
    pred=pipeline.predict(xte)
    return ('accuracy',round(float(accuracy_score(yte,pred)),4)) if is_class else ('MAE',round(float(mean_absolute_error(yte,pred)),3))


def main():
    if INDUSTRY not in CATALOG: raise ValueError(f'Unknown industry; choose {list(CATALOG)}')
    case=USE_CASE or CATALOG[INDUSTRY][0]
    if case not in CATALOG[INDUSTRY]: raise ValueError(f'For {INDUSTRY}, choose {CATALOG[INDUSTRY]}')
    if ROWS<30 or not 0<=NOISE<=2: raise ValueError('ROWS >= 30, NOISE between 0 and 2')
    df=add_missing(generate(INDUSTRY,case,ROWS,SEED,NOISE),MISSING_RATE,SEED)
    targets=TARGETS  # Shared catalog: used by both UI and generator.
    target=targets[case]; dest=Path(OUTPUT_DIR); dest.mkdir(parents=True,exist_ok=True); stem=f'{INDUSTRY}_{case}_{ROWS}'
    csv=dest/f'{stem}.csv'; js=dest/f'{stem}.json'; meta=dest/f'{stem}_metadata.json'; fig=dest/f'{stem}_preview.png'
    df.to_csv(csv,index=False); df.to_json(js,orient='records',indent=2,date_format='iso')
    metadata={'industry':INDUSTRY,'use_case':case,'rows':ROWS,'seed':SEED,'noise':NOISE,'missing_rate':MISSING_RATE,'target':target,
              'columns':df.dtypes.astype(str).to_dict(),'synthetic':True,'warning':'Simulated data with illustrative, not empirically validated, domain relationships.'}
    meta.write_text(json.dumps(metadata,indent=2))
    (dest/'ds_gen_catalog.json').write_text(json.dumps({k:list(v) for k,v in CATALOG.items()},indent=2))
    print('Industry/use case:',INDUSTRY,case,'| target:',target)
    print('Available cases:',CATALOG)
    try: display(df.head(8))
    except NameError: print(df.head(8).to_string(index=False))
    print('Missing cells:',int(df.isna().sum().sum()))
    fig_obj,ax=plt.subplots(figsize=(8,3.5)); df[target].value_counts().sort_index().plot.bar(ax=ax) if df[target].nunique()<=12 else df[target].hist(ax=ax,bins=30)
    ax.set_title(f'{INDUSTRY} / {case}: {target}'); fig_obj.tight_layout(); fig_obj.savefig(fig,dpi=120); plt.show(); plt.close(fig_obj)
    if RUN_MODEL:
       try: print('Baseline (illustrative):',run_baseline(df,target,SEED))
       except Exception as e: print('Baseline could not run:',type(e).__name__,str(e))
    zippath=dest/f'{stem}_kaggle_dataset.zip'
    with zipfile.ZipFile(zippath,'w',compression=zipfile.ZIP_DEFLATED) as z:
       for p in [csv,js,meta,fig]: z.write(p,arcname=p.name)
    print('Saved Kaggle outputs:')
    for p in [csv,js,meta,fig,zippath,dest/'ds_gen_catalog.json']: print(' ',p)
    return df

# Read live dropdown choices when Cell 2 is executed.
if 'industry_picker' in globals() and 'case_picker' in globals():
    INDUSTRY = industry_picker.value
    USE_CASE = case_picker.value
print(f"Generating: {INDUSTRY} / {USE_CASE} (target: {TARGETS[USE_CASE]})")
df=main()
